# Introduction
The goal of this notebook is to slowly build up to having full AGN simulation using agnSED and perturbations to account for the time domain.

## 1 Thin disk model
We will start by implementing a thin disk model with the gaussian random field perturbation s only. To simplify the code, we will start by building them without Caskade and without JAX. 

In [2]:
import os
from warnings import filters
import jax.numpy as jnp
import jax
from caskade import Param, forward
import numpy as np
from cosmographi.cosmology import Cosmology
from .base import TransientSource
from ..utils import flux
from ..utils.constants import Mpc_to_cm
from typing import Any
from ..utils.constants import c_m
from scipy.integrate import quad

2026-06-17 15:12:38.983343: W external/xla/xla/service/gpu/nvptx_compiler.cc:760] The NVIDIA driver's CUDA version is 12.2 which is older than the ptxas CUDA version (12.9.86). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.


ImportError: attempted relative import with no known parent package

In [ ]:
pi = np.pi
G = 6.6743 * 10e-11  # m3⋅kg−1⋅s−2
sigma =  5.670374419 * 10e-8
h = 6.62607015 * 10e-34 #m2 kg / s planck's constant
k = 1.380649 * 10e-23 # m2 kg s-2 K-1  boltzman constant
M_sun = 1.98847e30      # kg
# we assume R∗​=RISCO​=6GM​/c**2
#  Stephan-Boltzman constant (sigma) = 5.670374419 × 10⁻⁸ W⋅m⁻²⋅K⁻
def temp(R: float, mass: float, acc_rate: float, r_star: float) -> float:
    """ Return the temperature at distance <R> for an AGN of <mass> and accretion rate <acc_rate>"""
    first_prod = 3 * G * mass * acc_rate / 8 /pi / R**3 / sigma
    second_prod = 1 - (r_star / R) ** (1/2)
    return (first_prod * second_prod) ** (1/4)

def _integrand_funct(R: float, freq: float, mass: float, acc_rate: float, r_star: float) -> float:
    return R / (np.exp(h*freq/k/temp(R, mass, acc_rate, r_star)) - 1)

def flux_density(freq: float, inclination_angle: float, mass: float, acc_rate: float, r_star: float, luminosity_distance: float) -> float:
    """ Calculate the flux density at a specific frecuency for a specific agn"""
    first_prod = 4*pi*h*np.cos(inclination_angle)*freq**3/c_m**2 / luminosity_distance**2
    # assume R_out is 10**4 * r_star
    second_prod = quad(_integrand_funct, r_star, 10**4 * r_star, args=(freq,mass,acc_rate,r_star))[0]
    return first_prod * second_prod 


In [3]:
flux_density(c_m/(500e-9), 0, 10e8*M_sun, M_sun, 1.5e11, 0.01)

NameError: name 'flux_density' is not defined